In [2]:
"""
Heatmap KDE desde puntos densificados (GeoPackage).
Configura PATH, RES, BW, KERNEL abajo y ejecuta.
"""
import os, numpy as np, geopandas as gpd, rasterio
from rasterio.transform import from_bounds
from shapely import get_coordinates
from scipy.fft import rfft2, irfft2

# ===== CONFIG =====
POINTS_PATH = r"C:\Users\LAV\Desktop\Proyecto Vannia\Least-Cost-Path\output\session_20260527_204842_Refactored\puntos_rutas_25m.gpkg"
HEATMAP_RES_M = 30
HEATMAP_BANDWIDTH_M = 10000
HEATMAP_KERNEL = "quartic"  # quartic, gaussian, triangular, uniform
# ==================

OUTPUT_DIR = os.path.dirname(POINTS_PATH)
print(f"Cargando puntos: {POINTS_PATH}")
points_gdf = gpd.read_file(POINTS_PATH)
crs = points_gdf.crs
print(f"  CRS: {crs}  |  Puntos: {len(points_gdf):,}")

# Extent con margen = bandwidth
minx, miny, maxx, maxy = points_gdf.total_bounds
margin = HEATMAP_BANDWIDTH_M
minx -= margin; miny -= margin; maxx += margin; maxy += margin

width  = int(np.ceil((maxx - minx) / HEATMAP_RES_M))
height = int(np.ceil((maxy - miny) / HEATMAP_RES_M))
pixel_size_x = (maxx - minx) / width
pixel_size_y = (maxy - miny) / height
transform = from_bounds(minx, miny, maxx, maxy, width, height)

print(f"  Grid: {width}x{height} pix | Pixel: {pixel_size_x:.4f}x{pixel_size_y:.4f}m")

# Binning
print("Binning...")
coords = get_coordinates(points_gdf.geometry)
cols = ((coords[:, 0] - minx) / pixel_size_x).astype(np.int32)
rows = ((maxy - coords[:, 1]) / pixel_size_y).astype(np.int32)
valid = (cols >= 0) & (cols < width) & (rows >= 0) & (rows < height)
count_grid = np.zeros((height, width), dtype=np.float64)
np.add.at(count_grid, (rows[valid], cols[valid]), 1)
print(f"  Binnados: {valid.sum():,} / {len(coords):,}")

# Kernel
ps = (pixel_size_x + pixel_size_y) / 2.0
radius_px = int(np.ceil(HEATMAP_BANDWIDTH_M / ps))
y, x = np.ogrid[-radius_px:radius_px+1, -radius_px:radius_px+1]
u = np.sqrt(x**2 + y**2) * ps / HEATMAP_BANDWIDTH_M

if HEATMAP_KERNEL == "quartic":
    kernel = np.where(u <= 1, (15.0/16.0)*(1-u**2)**2, 0.0)
elif HEATMAP_KERNEL == "gaussian":
    kernel = np.exp(-0.5*u**2)
elif HEATMAP_KERNEL == "triangular":
    kernel = np.where(u <= 1, 1-u, 0.0)
elif HEATMAP_KERNEL == "uniform":
    kernel = np.where(u <= 1, 1.0, 0.0)
else:
    raise ValueError(f"Kernel: {HEATMAP_KERNEL}")
kernel = kernel / (np.sum(kernel) * ps**2)
print(f"  Kernel: {kernel.shape} px, bw={HEATMAP_BANDWIDTH_M/1000:.0f}km")

# FFT
print("Convolucion FFT...")
count_padded = np.pad(count_grid, pad_width=radius_px, mode='constant')
kp = np.zeros_like(count_padded)
kp[:kernel.shape[0], :kernel.shape[1]] = kernel
heatmap = irfft2(rfft2(count_padded) * rfft2(kp), s=count_padded.shape)
heatmap = heatmap[2*radius_px:2*radius_px+height, 2*radius_px:2*radius_px+width]
print(f"  Densidad: {heatmap.min():.6f} a {heatmap.max():.6f}")

# Guardar
tif_path = os.path.join(OUTPUT_DIR,
    f'heatmap_{HEATMAP_RES_M}m_{int(HEATMAP_BANDWIDTH_M/1000)}km_{HEATMAP_KERNEL}.tif')
with rasterio.open(tif_path, 'w', driver='GTiff',
    height=height, width=width, count=1, dtype=heatmap.dtype,
    crs=crs, transform=transform, compress='lzw') as dst:
    dst.write(heatmap, 1)

# Verificacion
with rasterio.open(tif_path) as src:
    print(f"\nVERIFICACION:")
    print(f"  Bounds: {src.bounds}")
    print(f"  Transform: {src.transform}")
    print(f"  CRS: {src.crs}")
print(f"\nGeoTIFF: {tif_path}")
print("LISTO.")

Cargando puntos: C:\Users\LAV\Desktop\Proyecto Vannia\Least-Cost-Path\output\session_20260527_204842_Refactored\puntos_rutas_25m.gpkg
  CRS: EPSG:32719  |  Puntos: 236,745,086
  Grid: 10109x6058 pix | Pixel: 29.9990x29.9983m
Binning...
  Binnados: 236,745,086 / 236,745,086
  Kernel: (669, 669) px, bw=10km
Convolucion FFT...
  Densidad: -0.000000 a 0.052608

VERIFICACION:
  Bounds: BoundingBox(left=279687.6383, bottom=6081385.0489, right=582947.6383, top=6263115.0489)
  Transform: | 30.00, 0.00, 279687.64|
| 0.00,-30.00, 6263115.05|
| 0.00, 0.00, 1.00|
  CRS: EPSG:32719

GeoTIFF: C:\Users\LAV\Desktop\Proyecto Vannia\Least-Cost-Path\output\session_20260527_204842_Refactored\heatmap_30m_10km_quartic.tif
LISTO.
